# Summary

Test calling a deployed agent

In [1]:
import os, sys
import json
import time
from datetime import datetime

from vertexai import agent_engines

import vertexai

# Import libraries from the Agent Development Kit
from google.adk.agents import Agent
from google.adk.artifacts import InMemoryArtifactService
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.sessions import VertexAiSessionService
from google.genai import types


# Set environment variables
utils_path = "utils/"
sys.path.insert(0, utils_path)
from authentication import ApiAuthentication
api_configs = ApiAuthentication(client="CCC")

# Initialize Vertex AI API once per session
# vertexai.init(project=os.environ["GOOGLE_CLOUD_PROJECT"],
#               location=os.environ["GOOGLE_CLOUD_LOCATION"],
#               staging_bucket=os.environ["STAGING_BUCKET"])



In [2]:
from search_tools import readVaiSearchResults
from search_tools import readGoogleSearchResults


## Call search tools

In [15]:
q1 = "What are the responsibilities of the board members of a California community college?"
q1 = "What are LEAP exams"
user_id = "U_123"

va_results = readVaiSearchResults(query=q1)

gs_results = readGoogleSearchResults(query=q1, user_id=user_id)


## Combine search contents

In [16]:
va_results.contents
gs_results.contents

contents = va_results.contents + gs_results.contents
contents

long_content_string = " ".join(contents)
print(len(long_content_string))


26369


## Call the synthesis agents

In [17]:
query = ("Use the following search results to synthesize an answer to this user query: {}?  "
         "Search results: {}.").format(q1,
                                       long_content_string)

query

'Use the following search results to synthesize an answer to this user query: What are LEAP exams.  Search results: Policy Priority Areas Resources by State Trending Topics Ed Note Blog Education Commission of the States is the trusted source for comprehensive knowledge and unbiased resources on education policy issues ranging from early learning through postsecondary education. Subscribe to our publications and stay informed. Need more information? Contact one of our policy experts. Assessments serve a variety of purposes and take multiple forms depending on what they are measuring and how they are used. They are administered at various stages of a student’s education journey to evaluate a student’s knowledge of specific academic subjects or mastery of skills. Assessment systems can provide data that is valuable for education stakeholders including: Students and families, teachers and school leaders, district and state Leaders. If you\'re just getting started with student assessment, 

In [18]:

# Retreive agent
resource_name = "projects/eternal-bongo-435614-b9/locations/us-central1/reasoningEngines/2582084310576136192"
agent_engine = agent_engines.get(resource_name)

# Establish session
session = agent_engine.create_session(user_id=user_id)

# Get agent response
result = agent_engine.stream_query(message=query,
                                   session_id=session["id"],
                                   user_id=user_id)

# Put results into a dictionary for later access
events = []
for event in result:
    events.append(event)

## Provide a synthesized answer

In [19]:
events

for event in events:
    if type(event) == dict and "content" in event.keys():
        print(event["content"]["parts"][0]["text"])

## LEAP (Limited Examination and Appointment Program) Exams: A Comprehensive Overview

LEAP exams are an alternative pathway for individuals with disabilities to secure employment within the
California Community Colleges system and other California State Civil Service positions.
Administered by the California Community Colleges Chancellor's Office, LEAP streamlines the hiring process for
eligible candidates, focusing on practical skills and on-the-job performance rather than traditional examination
formats.

### Key Features of LEAP
*   **Targeted Support:** LEAP is specifically designed to assist individuals with disabilities in obtaining
State employment. Eligibility requires certification from the California Department of Rehabilitation (DOR).
*   **Alternative Assessment:** Instead of standard civil service exams, LEAP employs a Job Examination Period
(JEP), which allows candidates to demonstrate their abilities in a real-world work setting.
*   **Job Examination Period (JEP):** Th

## Show links and organizations

In [25]:
print("Here are additional resources\n")
print("Organizations who have helpful content")
print(va_results.organizations)


print("\nUseful links")
print(va_results.uris + gs_results.uris)

print()

Here are additional resources

Organizations who have helpful content
[{'role': 'Education Commission of the States is the essential, indispensable member of any team addressing education policy at the state level. ', 'uri': 'https://www.ecs.org', 'name': 'Education Commission of the States'}, {'name': 'Community College Research Center, Columbia University', 'uri': 'https://ccrc.tc.columbia.edu', 'role': 'The Community College Research Center (CCRC) is an independent organization in the field of community college research and reform.'}, {'name': 'California Community Colleges', 'role': 'Informational site for all California community colleges.', 'uri': 'https://www.cccco.edu'}]

Useful links
['https://www.cccco.edu/About-Us/job-opportunities', 'https://www.ecs.org/moving-from-single-to-multiple-measures-for-college-course-placement/#addtoany', 'https://www.ecs.org/moving-from-single-to-multiple-measures-for-college-course-placement/#dropdown', 'https://www.ecs.org/moving-from-single-t

####################

## Retrieve a deployed agent

In [2]:
dir(agent_engines)

# get a list of agents
for agent in agent_engines.list():
    print(agent.resource_name)

# Delete if needed
# for agent in agent_engines.list():
#     agent_engines.delete(resource_name=agent.resource_name,
#                          force=True)


projects/1062597788108/locations/us-central1/reasoningEngines/2582084310576136192
projects/1062597788108/locations/us-central1/reasoningEngines/6068151897137610752
projects/1062597788108/locations/us-central1/reasoningEngines/5673523979789271040


In [3]:
# os.getenv("AGENT_ENGINE_ID")

In [4]:
vertexai_resource_name = "projects/1062597788108/locations/us-central1/reasoningEngines/5673523979789271040"
search_resource_name = "projects/1062597788108/locations/us-central1/reasoningEngines/6068151897137610752"
resource_name = vertexai_resource_name


agent_engine2 = agent_engines.get(resource_name)
print("Testing a local deployment of agent: {}".format(agent_engine2.name))
start_time = datetime.now()
print("Time: {}".format(start_time.strftime("%b %-d, %Y, %-H:%M:%S %p")))


session = agent_engine2.create_session(user_id="u_123")
print("Session details: {}".format(session))

q1 = "What are the responsibilities of the board members of a California community college?"
test_result = agent_engine2.stream_query(message=q1,
                                        session_id=session["id"],
                                        user_id="U_123")
events = []
for event in test_result:
    events.append(event)

    print("Event Author: {}".format(event["author"]))
    print("Text: {} ...".format(event['content']["parts"][0]["text"][:750]))
    print()


Testing a local deployment of agent: 5673523979789271040
Time: Jun 18, 2025, 5:57:27 AM
Session details: {'lastUpdateTime': 1750251447.972632, 'id': '4860998630758154240', 'state': {}, 'events': [], 'appName': '5673523979789271040', 'userId': 'u_123'}


In [21]:

# Retreive an ADK agent
# agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))




# dir(agent_engine)
# Opt 1: Create a session using agent_engine
session = agent_engine.create_session(user_id="u_123")



#### Option 2
# session_service = VertexAiSessionService(project=os.environ["GOOGLE_CLOUD_PROJECT"],
#                                          location=os.environ["GOOGLE_CLOUD_LOCATION"])
#
# # type(agent_engine)
# session_service_active1 = agent_engine.create_session(user_id="u_123")


In [11]:
print(session)
session['id']

{'userId': 'u_123', 'appName': '9020261452878970880', 'id': '8902275608881397760', 'events': [], 'state': {}, 'lastUpdateTime': 1750173511.784099}


'8902275608881397760'

In [9]:
type(agent_engine)
# dir(agent_engines)
# agent_engine.operation_schemas()
# dir(session)


vertexai.agent_engines._agent_engines.AgentEngine

In [6]:
agent_engine

resource name: projects/1062597788108/locations/us-central1/reasoningEngines/9020261452878970880

In [8]:
# # from google.adk.sessions import VertexAiSessionService
# #
# app_name=os.getenv("AGENT_ENGINE_ID")
# user_id="u_123"
#
# # Create the ADK runner with VertexAiSessionService
# session_service = VertexAiSessionService(project=os.environ["GOOGLE_CLOUD_PROJECT"],
#                                          location=os.environ["GOOGLE_CLOUD_LOCATION"])
# runner = Runner(
#     # agent=root_agent,
#     agent=agent_engine,
#     app_name=app_name,
#     session_service=session_service)
#
#
# # Helper method to send query to the runner
# def call_agent(query, session_id, user_id):
#   content = types.Content(role='user', parts=[types.Part(text=query)])
#   events = runner.run(
#       user_id=user_id, session_id=session_id, new_message=content)
#
#   for event in events:
#       if event.is_final_response():
#           final_response = event.content.parts[0].text
#           print("Agent Response: ", final_response)
#
# q1 = "What are LEAP exams?"
# # q2 = "What are the responsibilities of the board members of a California community college?"
#
# test_result2 = call_agent(query=q1, session_id=user_id="U_123")

In [9]:
# type(agent_engine)
# dir(session_service)

# session.keys()

## Test the agent with one query

In [22]:
q1 = "What are LEAP exams?"
q1 = "What are the responsibilities of the board members of a California community college?"

test_result = agent_engine.stream_query(message=q1,
                                        session_id=session["id"],
                                        user_id="U_123")


# test_result = agent_engine.async_stream_query(message=q1, user_id="U_123")

events = []
for event in test_result:
    # print(event)
    events.append(event)



In [23]:
for event in events:
    print(type(event))
    print(event.keys())
    print(event['content'].keys())
    # print(event['content']['parts'])
    # print(event['grounding_metadata'])
    print(event['author'])
    print(event['actions'])
    print()




<class 'dict'>
dict_keys(['content', 'grounding_metadata', 'usage_metadata', 'invocation_id', 'author', 'actions', 'id', 'timestamp'])
dict_keys(['parts', 'role'])
search_assistant
{'state_delta': {}, 'artifact_delta': {}, 'requested_auth_configs': {}}



In [31]:
event['content']["parts"][0]["text"][:750]

"The responsibilities of board members of a California community college are multifaceted, revolving around governance, policy setting, and ensuring the well-being of the college district. Here's a breakdown of their key duties:\n\n**1. Governance and Policy:**\n\n*   **Policy Setting:** Board members are responsible for establishing policies that define the institutional mission and set standards for college operations, ensuring these policies align with legal and ethical guidelines.\n*   **Strategic Direction:** They set the college's strategic direction, focusing on the future learning needs of their communities.\n*   **Monitoring Performance:** They monitor the institution's performance, holding the colleges accountable for achieving student s"

In [7]:
for event in events:
    if event["author"] == "rag_search_agent":
        print(event["author"])
        e1 = event
        for entry in event["content"]["parts"]:
            try:
                print(entry["text"])
            except:
                pass

In [8]:
events

[]

In [ ]:
e1

In [ ]:
e1.keys()

for key in e1.keys():
    # print(key)
    # print(e1[key])
    # print()
    if key == "grounding_metadata":
        print(e1[key].keys())
        print()
        for gc in e1[key]:
            # print(gc)
            # print(e1[key][gc])
            # print()
            #
            # get urls

            # get metadata
            if gc == "grounding_chunks":
                for gcli in e1[key]["grounding_chunks"]:
                    # print(gcli["retrieved_context"])

                    for gclikey in gcli["retrieved_context"].keys():

                        for gcl4key in gcli["retrieved_context"][gclikey].keys():
                            print(gcl4key)
                            print(type(gcli["retrieved_context"][gclikey][gcl4key]))
                            print("====")

                        # print(gclikey)
                        # print(type(gcli["retrieved_context"][gclikey]))
                        # print(gcli["retrieved_context"][gclikey])
                        #
                        # print("====")

    #


In [ ]:
import re

finds_list = []
for item in e1["grounding_metadata"]["grounding_chunks"]:
    # print(type(item["retrieved_context"]["rag_chunk"]["text"]))
    # print(item["retrieved_context"]["rag_chunk"]["text"])
    # print("===")

    pats = {"title": r"structData title (.*)", "page_url": r"structData page_url (.*)"}


    for patkey in pats:

        patfinds = re.findall(pats[patkey], item["retrieved_context"]["rag_chunk"]["text"])
        if patfinds:
            for patfind in patfinds:
                finds_list.append({patkey: patfind})



In [ ]:
e1["grounding_metadata"]["grounding_chunks"][3]

In [ ]:
finds_list

# t2 = finds[1]
# dir(t2)
#
# t2.string

# find_between(item["retrieved_context"]["rag_chunk"]["text"], "StructData", "StructData")

In [ ]:
from google.cloud import discoveryengine_v1alpha as discoveryengine
from google.api_core.client_options import ClientOptions
import functions_framework
import json

# Your Project, Location, and Data Store ID
PROJECT_ID = "your-gcp-project-id"
LOCATION = "global"  # Or your specific location
DATA_STORE_ID = "your-data-store-id"

@functions_framework.http
def search_with_metadata(request):
    """
    Receives a query from a Vertex Agent and returns a response
    with structured metadata.
    """
    request_json = request.get_json(silent=True)
    query = request_json['sessionInfo']['parameters']['query'] # Adjust based on how agent sends params

    if not query:
        return json.dumps({"fulfillment_response": {"messages": [{"text": {"text": ["Please provide a search query."]}}]}})

    client = discoveryengine.SearchServiceClient(
        client_options=ClientOptions(api_endpoint=f"{LOCATION}-discoveryengine.googleapis.com")
    )
    serving_config = client.serving_config_path(
        project=PROJECT_ID, location=LOCATION,
        data_store=DATA_STORE_ID, serving_config="default_config"
    )

    search_request = discoveryengine.SearchRequest(
        serving_config=serving_config,
        query=query,
        page_size=3 # Get top 3 results
    )

    response = client.search(search_request)

    # Format the response for the agent
    final_response = "Here are the top results for your query:\n\n"
    for i, result in enumerate(response.results):
        doc_data = result.document.struct_data
        final_response += f"--- Result {i+1} ---\n"
        final_response += f"Snippet: ...{result.document.derived_struct_data['snippets'][0]['snippet']}...\n"
        final_response += f"Author: {doc_data.get('author', 'N/A')}\n"
        final_response += f"Source URL: {doc_data.get('source_url', 'N/A')}\n"
        final_response += f"Document ID: {result.document.id}\n\n"


    # Create the response JSON for Vertex AI Agent Builder
    response_json = {
        "fulfillment_response": {
            "messages": [
                {
                    "text": {
                        "text": [final_response]
                    }
                }
            ]
        }
    }

    return json.dumps(response_json)

In [ ]:
# agent_engines.get("4226237373903536128")
# dir(agent_engines)

# dir(agent_engine)
#
# test_session = agent_engine.get_session(user_id="U_123", session_id="4226237373903536128")
#
# dir(test_session)
#
# dir(test_session.items())



In [ ]:
# for event in events:
#     if event["author"] == "synthesis_agent":
#         for entry in event["content"]["parts"]:
#             print(entry["text"])
#
for event in events:
#     # if event["author"] == "synthesis_agent":
    print(event["author"])
    for entry in event["content"]["parts"]:
        try:
            print(entry["text"])
        except:
            pass
#
    print()
    print()


# events


## Test the agent with multiple queries in a session

In [ ]:
def pretty_print_event(event):
    """Pretty prints an event with truncation for long content."""
    if "content" not in event:
        print(f"[{event.get('author', 'unknown')}]: {event}")
        return

    author = event.get("author", "unknown")
    parts = event["content"].get("parts", [])

    for part in parts:
        if "text" in part:
            text = part["text"]
            # Truncate long text to 200 characters
            if len(text) > 200:
                text = text[:197] + "..."
            print(f"[{author}]: {text}")
        elif "functionCall" in part:
            func_call = part["functionCall"]
            print(f"[{author}]: Function call: {func_call.get('name', 'unknown')}")
            # Truncate args if too long
            args = json.dumps(func_call.get("args", {}))
            if len(args) > 100:
                args = args[:97] + "..."
            print(f"  Args: {args}")
        elif "functionResponse" in part:
            func_response = part["functionResponse"]
            print(f"[{author}]: Function response: {func_response.get('name', 'unknown')}")
            # Truncate response if too long
            response = json.dumps(func_response.get("response", {}))
            if len(response) > 100:
                response = response[:97] + "..."
            print(f"  Response: {response}")

agent_engine = agent_engines.get(os.getenv("AGENT_ENGINE_ID"))
print(f"Agent ID: Id='{agent_engine.resource_name}'")

# Create a session
user_id = "u_123"
session = agent_engine.create_session(user_id=user_id)
print(f"Session created: User='{user_id}', Session='{session['id']}'")

# Create some queries
queries = [
    "Hi, how are you?",
    "What are LEAP exams?",
    "What is the California Community Colleges Chancellor’s Office doing to ensure accessibility of its websites"
]

# Look for events for each query
for query in queries:
    print(f"\n[user]: {query}")
    for event in agent_engine.stream_query(
        user_id=user_id,
        session_id=session["id"],
        message=query,
    ):
        print("We got here by finding an event")
        pretty_print_event(event)
        print(session.get(session["id"]))


## Look at the sessions

In [ ]:
agent_engine.list_sessions(user_id=user_id)

## Create - This stuff doesn't work with the deployed agent but does work if the deployed agent is called directly from code

In [ ]:

def send_query_to_agent(agent, query, user_id):
    """Sends a query to the specified agent and prints the response.

    Args:
        agent: The agent to send the query to.
        query: The query to send to the agent.

    Returns:
        A tuple containing the elapsed time (in milliseconds) and the final response from the agent.
    """

    # Create a new session - if you want to keep the history of interruction you need to move the
    # creation of the session outside of this function. Here we create a new session per query
    session = session_service.create_session(app_name=AGENT_APP_NAME,
                                             user_id=user_id,)
    # Create a content object representing the user's query
    print('\nUser Query: ', query)
    content = types.Content(role='user', parts=[types.Part(text=query)])

    # Start a timer to measure the response time
    start_time = time.time()

    # Create a runner object to manage the interaction with the agent
    runner = Runner(app_name=AGENT_APP_NAME, agent=agent, artifact_service=artifact_service, session_service=session_service)

    # Run the interaction with the agent and get a stream of events
    events = runner.run(user_id=user_id, session_id=session.id, new_message=content)

    final_response = None
    elapsed_time_ms = 0.0

    # Loop through the events returned by the runner
    for _, event in enumerate(events):

        is_final_response = event.is_final_response()

        if not event.content:
             continue

        if is_final_response:
            end_time = time.time()
            elapsed_time_ms = round((end_time - start_time) * 1000, 3)

            print("-----------------------------")
            print('>>> Inside final response <<<')
            print("-----------------------------")
            final_response = event.content.parts[0].text # Get the final response from the agent
            print(f'Agent: {event.author}')
            print(f'Response time: {elapsed_time_ms} ms\n')
            print(f'Final Response:\n{final_response}')
            print("----------------------------------------------------------\n")

    return elapsed_time_ms, final_response



In [ ]:
MODEL='gemini-2.0-flash-001'

# # Create a basic agent with instructions amd greeting only
# basic_agent = Agent(model=MODEL,
#     name="agent_basic",
#     description="This agent responds to inquiries about its creation by stating it was built using the Google Agent Framework.",
#     instruction="If they ask you how you were created, tell them you were created with the Google Agent Framework.",
#     generate_content_config=types.GenerateContentConfig(temperature=0.2),
# )

############## Import the agent code from the python file used to create the deployed agent
rag_path = "ccc_chatbot/"
sys.path.insert(0, rag_path)
print(os.listdir(rag_path))
from agent import root_agent

basic_agent = root_agent

# Get the agent
# basic_agent = agent_engines.get(os.getenv("AGENT_ENGINE_ID2"))

# Create session and memory services
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# AGENT_APP_NAME = basic_agent.resource_name
AGENT_APP_NAME = 'agent_basic'
user_id = "u_123"

# Send a single query to the agent
send_query_to_agent(basic_agent, "How are you today", user_id)


In [ ]:
from google.adk.agents import SequentialAgent
from google.adk.agents import LlmAgent

basic_agent = agent_engines.get(os.getenv("AGENT_ENGINE_ID2"))

code_pipeline_agent = SequentialAgent(
    name="run_other_agents",
    # sub_agents=[basic_agent],
    tools=[basic_agent],
    description="Wrapper for the basic agent",
)

# Create session and memory services
session_service = InMemorySessionService()
artifact_service = InMemoryArtifactService()

# AGENT_APP_NAME = basic_agent.resource_name
AGENT_APP_NAME = 'agent_basic'
user_id = "u_123"

# Send a single query to the agent
send_query_to_agent(code_pipeline_agent, "How are you today", user_id)